# 🧊 TRELLIS 3D Generator – خلية واحدة (إصدار لطيف)
⚠️ **تنبيه:** قبل تشغيل الخلية، تأكد من تفعيل GPU:
**Runtime → Change runtime type → Hardware accelerator: GPU (T4)**

الخلية ستتوقف بأمان إذا لم تجد GPU وتعطيك التعليمات.

In [ ]:
# =============================================
# TRELLIS 3D Generator - خلية واحدة (لطيفة)
# =============================================
import os, sys, subprocess, zipfile, io, requests, tempfile, traceback, importlib.metadata
import torch, numpy as np
from PIL import Image
import gradio as gr

# 1. فحص GPU
print("🔍 فحص GPU...")
if not torch.cuda.is_available():
    print("❌ لم يتم العثور على GPU!")
    print("الرجاء: Runtime → Change runtime type → Hardware accelerator → GPU (مثلاً T4)")
    print("ثم Runtime → Restart runtime، وأعد تشغيل هذه الخلية.")
    exit(0)

!nvidia-smi
gpu_name = subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader")
print(f"✅ GPU: {gpu_name}")

SUPPORTED_GPU = ["A100", "A6000", "3090", "4090", "L4", "L40", "H100"]
USE_FLASH_ATTN = any(g in gpu_name for g in SUPPORTED_GPU)
print("✨ Flash Attention 2 مدعوم" if USE_FLASH_ATTN else "ℹ️  يستخدم SDPA.")

# 2. فحص وتثبيت الحزم (باستخدام importlib.metadata)
def is_installed(pkg):
    try:
        importlib.metadata.version(pkg)
        return True
    except importlib.metadata.PackageNotFoundError:
        return False

print("\n🔍 فحص الحزم الأساسية...")
base_pkgs = ["trimesh", "pygltflib", "viser", "omegaconf", "opencv-python", "imageio", "gradio", "huggingface_hub"]
for pkg in base_pkgs:
    if not is_installed(pkg):
        !pip install -q {pkg}
        print(f"✔️ {pkg}")
    else:
        print(f"✅ {pkg} (موجود)")

# spconv
if not is_installed("spconv"):
    cuda_ver = torch.version.cuda.replace(".", "")
    try:
        !pip install -q spconv-cu{cuda_ver}
    except:
        !pip install -q spconv
    print("✔️ spconv")
else:
    print("✅ spconv (موجود)")

# Flash Attention 2 (اختياري)
if USE_FLASH_ATTN and not is_installed("flash-attn"):
    print("⚡ تثبيت Flash Attention 2...")
    !pip install -q ninja packaging
    !pip install flash-attn --no-build-isolation 2>&1 | tail -5
    import flash_attn
    print("✅ flash-attn")

# 3. تحميل مكتبة TRELLIS من GitHub (ZIP)
ZIP_URL = "https://codeload.github.com/JeffreyXiang/TRELLIS/zip/refs/heads/main"
TARGET_DIR = "/content/TRELLIS-main"
if not os.path.isdir(TARGET_DIR):
    print("⬇️  تحميل المستودع...")
    try:
        r = requests.get(ZIP_URL, timeout=30)
        r.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            z.extractall("/content")
        print("✅ تم التنزيل وفك الضغط.")
    except:
        print("❌ فشل التحميل. يرجى رفع الملف يدوياً.")
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded.keys():
            with zipfile.ZipFile(io.BytesIO(uploaded[fn])) as z:
                z.extractall("/content")
            print("✅ تم فك الضغط.")

if not os.path.isdir(TARGET_DIR):
    raise FileNotFoundError("لم يتم العثور على مجلد TRELLIS-main.")
sys.path.insert(0, TARGET_DIR)
import trellis
print("✅ استيراد trellis ناجح.")

# إعداد الانتباه
if USE_FLASH_ATTN:
    try:
        import flash_attn
        attn_impl = "flash_attention_2"
        print("🔧 Flash Attention 2 مفعل.")
    except:
        attn_impl = "sdpa"
        print("🔧 استخدام SDPA.")
else:
    attn_impl = "sdpa"
    print("🔧 SDPA مفعل.")

# 4. تحميل النموذج
from trellis.pipelines import TrellisImageTo3DPipeline
try:
    pipeline = TrellisImageTo3DPipeline.from_pretrained(
        "JeffreyXiang/TRELLIS-image-large",
        torch_dtype=torch.float16,
        attn_implementation=attn_impl
    ).to("cuda")
    print("✅ تم تحميل خط الأنابيب.")
except Exception as e:
    print(f"⚠️ خطأ: {e}\nمحاولة بدون attn_implementation...")
    pipeline = TrellisImageTo3DPipeline.from_pretrained(
        "JeffreyXiang/TRELLIS-image-large",
        torch_dtype=torch.float16
    ).to("cuda")
    print("✅ تم تحميل خط الأنابيب بالوضع الافتراضي.")

# 5. دالة التوليد
def generate_3d(image_input, seed, cfg_scale):
    if image_input is None:
        raise gr.Error("يرجى رفع صورة.")
    image = Image.fromarray(image_input).convert("RGB") if isinstance(image_input, np.ndarray) else image_input.convert("RGB")
    try:
        try:
            outputs = pipeline.run(image, seed=int(seed), cfg_scale=float(cfg_scale))
        except TypeError:
            outputs = pipeline.run(image, seed=int(seed))
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"فشل التوليد: {e}")
    mesh = outputs.get('mesh')
    if mesh is None:
        raise gr.Error("لم يتم توليد شبكة.")
    tmpdir = tempfile.mkdtemp()
    glb_path = os.path.join(tmpdir, "model.glb")
    mesh.export(glb_path)
    return glb_path, glb_path

# 6. واجهة Gradio
with gr.Blocks(title="TRELLIS 3D", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🧊 TRELLIS – تحويل الصورة إلى مجسم ثلاثي الأبعاد
    يدعم Flash Attention 2 تلقائياً. أرفع صورة واضغط على **توليد**.
    """)
    with gr.Row():
        with gr.Column():
            image_input = gr.Image(label="الصورة المدخلة", type="pil", image_mode="RGB")
            with gr.Accordion("⚙️ إعدادات متقدمة", open=False):
                seed = gr.Number(value=42, label="البذرة (Seed)", precision=0)
                cfg_scale = gr.Slider(minimum=1.0, maximum=20.0, value=7.5, step=0.5, label="CFG Scale")
            generate_btn = gr.Button("🚀 توليد", variant="primary")
        with gr.Column():
            model_output = gr.Model3D(label="النموذج الثلاثي الأبعاد")
            download_btn = gr.File(label="تحميل ملف GLB")
    generate_btn.click(fn=generate_3d, inputs=[image_input, seed, cfg_scale], outputs=[model_output, download_btn])

print("🌐 تشغيل واجهة Gradio...")
demo.queue(max_size=3).launch(share=True, server_port=7860, show_error=True)